# EDA — écarts entre baccalauréats et sélectivité

**Session de travail du 29 août 2026.** Suite du carnet consacré au label.

Le carnet précédent a établi la cible : sa définition, ses bornes, sa
distribution. Celui-ci répond à deux questions qui en découlent :

> 1. À formation égale, quel écart sépare un bachelier général d'un bachelier
>    professionnel — et cet écart est-il le même partout ?
> 2. Quelles variables expliquent le taux d'admission, et lesquelles puis-je
>    réellement utiliser au moment où mon système doit répondre ?

La seconde question est la plus importante, et je la traite en dernier : c'est
elle qui décide de ce qui entre ou non dans le modèle.

In [ ]:
import sys
from pathlib import Path

import matplotlib.pyplot as plt
import polars as pl

sys.path.insert(0, str(Path.cwd().parent / "src"))
from edumatch.config import load_settings

settings = load_settings("prod")
RAW = settings.raw_dir / "parcoursup"


def charger(annee: int) -> pl.DataFrame:
    return pl.read_csv(
        RAW / f"parcoursup_{annee}.csv",
        separator=";",
        encoding="utf8-lossy",
        infer_schema_length=2000,
    )


def taux(df: pl.DataFrame, suffixe: str) -> pl.Expr:
    """Taux d'admission d'une cellule, borné à 1 comme décidé au carnet précédent."""
    return (pl.col(f"prop_tot_{suffixe}") / pl.col(f"nb_voe_pp_{suffixe}")).clip(0, 1)


df = charger(2025)
print("session 2025 :", df.shape)

## 1. L'écart entre baccalauréats, à formation égale

Comparer la moyenne des bacheliers généraux à celle des bacheliers
professionnels sur l'ensemble du catalogue ne prouverait rien : les deux publics
ne visent pas les mêmes formations, et l'écart observé mélangerait la
sélectivité des formations avec les choix d'orientation.

Je compare donc **la même formation avec elle-même** : parmi celles qui reçoivent
des vœux des deux profils, quel écart de taux d'admission ?

In [ ]:
apparie = (
    df.filter((pl.col("nb_voe_pp_bg") > 0) & (pl.col("nb_voe_pp_bp") > 0))
    .with_columns([taux(df, "bg").alias("t_bg"), taux(df, "bp").alias("t_bp")])
    .with_columns((pl.col("t_bg") - pl.col("t_bp")).alias("ecart"))
)

print("formations recevant des vœux des deux profils :", apparie.height)
print()

e = apparie["ecart"]
print("écart de taux, bac général moins bac professionnel, dans la même formation")
print(f"  moyenne   {e.mean():>+7.3f}")
print(f"  médiane   {e.median():>+7.3f}")
for q in (0.10, 0.25, 0.75, 0.90):
    print(f"  q{int(q * 100):<3}      {e.quantile(q):>+7.3f}")
print()
print(f"  bac général avantagé : {(e > 0).sum():>6}  ({(e > 0).mean() * 100:.1f} %)")
print(f"  bac pro avantagé     : {(e < 0).sum():>6}  ({(e < 0).mean() * 100:.1f} %)")
print(f"  écart > 20 points    : {(e > 0.20).sum():>6}  ({(e > 0.20).mean() * 100:.1f} %)")

In [ ]:
fig, ax = plt.subplots(figsize=(9, 4))
ax.hist(apparie["ecart"].to_list(), bins=60, color="#4C72B0", edgecolor="white", linewidth=0.4)
ax.axvline(0, color="#C44E52", linewidth=1.4, label="écart nul")
ax.axvline(e.median(), color="#55A868", linewidth=1.4, linestyle="--", label=f"médiane {e.median():+.3f}")
ax.set_xlabel("taux bac général − taux bac professionnel, dans la même formation")
ax.set_ylabel("nombre de formations")
ax.set_title("Écart d'accès entre baccalauréats, comparaison appariée — session 2025")
ax.legend()
ax.grid(axis="y", alpha=0.25)
plt.tight_layout()
plt.show()

**Ce que j'en conclus.** L'écart médian est de **+8,5 points** en faveur du
bachelier général. Pris seul, ce chiffre serait trompeur, et c'est le piège que
je veux éviter : la distribution est **largement étalée de part et d'autre de
zéro**.

- 63,9 % des formations avantagent le bachelier général ;
- **26,8 % avantagent le bachelier professionnel** ;
- un tiers des formations présentent un écart supérieur à 20 points.

Autrement dit, il n'existe pas *un* désavantage du bac professionnel, mais une
**forte hétérogénéité** selon les formations. Annoncer « le bac professionnel est
désavantagé » serait une conclusion paresseuse, contredite par un quart des
observations.

C'est précisément ce qui fonde l'utilité de mon système : la question pertinente
pour un lycéen n'est pas « suis-je désavantagé ? » mais « **où** le suis-je, et
où ne le suis-je pas ? ».

## 2. Où se situe la coupure : la sélectivité par filière

Puisque l'écart varie fortement, je cherche ce qui le structure. La filière est
le premier candidat.

In [ ]:
par_filiere = (
    apparie.group_by("fili")
    .agg(
        pl.len().alias("n"),
        pl.col("t_bg").median().alias("mediane_bg"),
        pl.col("t_bp").median().alias("mediane_bp"),
        pl.col("ecart").median().alias("ecart_median"),
    )
    .filter(pl.col("n") >= 100)
    .sort("ecart_median")
)

print(f"{'filière':<24}{'n':>6}{'méd. bg':>10}{'méd. bp':>10}{'écart':>10}")
for filiere, n, m_bg, m_bp, ecart in par_filiere.iter_rows():
    print(f"{filiere[:24]:<24}{n:>6}{m_bg:>10.3f}{m_bp:>10.3f}{ecart:>+10.3f}")

In [ ]:
fig, ax = plt.subplots(figsize=(9, 5))
libelles = [r[0][:22] for r in par_filiere.iter_rows()]
valeurs = [r[4] for r in par_filiere.iter_rows()]
couleurs = ["#C44E52" if v > 0.20 else "#DD8452" if v > 0.10 else "#55A868" for v in valeurs]

ax.barh(libelles, valeurs, color=couleurs)
ax.set_xlabel("écart médian de taux d'admission (bac général − bac professionnel)")
ax.set_title("Écart d'accès par filière — session 2025")
ax.grid(axis="x", alpha=0.25)
plt.tight_layout()
plt.show()

**Ce que j'en conclus.** La filière structure l'écart de façon très nette, et
trois lectures s'imposent.

**Le BTS est neutre** : 1,1 point d'écart sur 5 231 formations. C'est la voie
naturelle du baccalauréat professionnel, et l'accès y est effectivement
comparable. C'est un point important à souligner : le système d'orientation n'est
pas uniformément fermé.

**Quatre filières affichent une médiane de taux nulle pour le bac
professionnel** : CPGE, BUT, PASS, et presque Licence-LAS. Il faut lire cette
valeur pour ce qu'elle dit — dans **plus de la moitié** de ces formations, un
bachelier professionnel qui formule un vœu ne reçoit **aucune** proposition. Ce
n'est pas « moins souvent », c'est aucune.

**L'IFSI est le cas le plus notable** : 33,2 points d'écart, deuxième rang
derrière la CPGE. Ce n'est pourtant pas une filière d'élite mais une formation
professionnalisante, ce qui rend l'écart d'autant plus frappant.

Je tiens à formuler ce que ces chiffres ne disent pas. Ils ne démontrent aucune
discrimination : une CPGE qui n'admet pas de bachelier professionnel peut avoir
des raisons pédagogiques défendables, tenant aux prérequis de la formation. Mon
système n'a pas vocation à juger ces pratiques, mais à **les rendre visibles
avant que le vœu ne soit formulé** — pour qu'un lycéen ne dépense pas un vœu sur
une porte statistiquement fermée, et qu'il identifie celles qui lui sont
réellement ouvertes.

## 3. La tension, et la question qui décide de tout

Je cherche maintenant ce qui explique le taux d'admission lui-même. Le candidat
le plus évident est la **tension** : le nombre de vœux reçus rapporté à la
capacité d'accueil.

In [ ]:
tension = df.filter((pl.col("capa_fin") > 0) & (pl.col("voe_tot") > 0)).with_columns(
    [
        (pl.col("voe_tot") / pl.col("capa_fin")).alias("tension"),
        taux(df, "bg").alias("t_bg"),
    ]
)

t = tension["tension"]
print(f"formations avec capacité et vœux renseignés : {tension.height}")
print(f"tension : médiane {t.median():.1f}   q10 {t.quantile(0.10):.1f}   q90 {t.quantile(0.90):.1f}   max {t.max():.0f}")
print()

quintiles = tension.with_columns(
    pl.col("tension").qcut(5, labels=["q1 faible", "q2", "q3", "q4", "q5 forte"]).alias("groupe")
)
resume = (
    quintiles.group_by("groupe")
    .agg(
        pl.len().alias("n"),
        pl.col("tension").median().alias("tension_mediane"),
        pl.col("t_bg").median().alias("taux_median"),
    )
    .sort("groupe")
)
print(f"{'quintile de tension':<20}{'n':>6}{'tension méd.':>14}{'taux méd. bg':>14}")
for groupe, n, tens, tx in resume.iter_rows():
    print(f"{groupe:<20}{n:>6}{tens:>14.1f}{tx:>14.3f}")

In [ ]:
fig, ax = plt.subplots(figsize=(8, 4))
groupes = [r[0] for r in resume.iter_rows()]
taux_med = [r[3] for r in resume.iter_rows()]
ax.bar(groupes, taux_med, color="#4C72B0")
for i, v in enumerate(taux_med):
    ax.text(i, v + 0.02, f"{v:.3f}", ha="center", fontsize=9)
ax.set_ylabel("taux d'admission médian (bac général)")
ax.set_xlabel("quintile de tension (vœux par place)")
ax.set_title("Le taux d'admission décroît strictement avec la tension — session 2025")
ax.set_ylim(0, 1.1)
ax.grid(axis="y", alpha=0.25)
plt.tight_layout()
plt.show()

**Ce que j'en conclus, et c'est le résultat décisif de ce carnet.**

La relation est **strictement monotone** : du quintile le moins tendu au plus
tendu, le taux d'admission médian passe de 1,000 à 0,182. La tension médiane est
de 11,4 vœux par place. C'est, de loin, le prédicteur le plus puissant que
j'observe.

Et c'est exactement pour cela qu'il faut s'arrêter avant de s'en réjouir, en
posant la seule question qui compte :

> `voe_tot`, le nombre total de vœux reçus par une formation en 2025, est-il
> connu **au moment où un lycéen formule ses vœux** en 2025 ?

**Non.** Ce nombre n'existe qu'une fois que tous les candidats ont formulé leurs
vœux. Mon meilleur prédicteur est donc inutilisable en l'état.

Je tiens à nommer précisément la nature du problème, parce qu'elle est subtile.
Ce n'est pas une fuite temporelle au sens strict : `voe_tot` précède bien la
décision d'admission que je cherche à prédire. C'est une **fuite fonctionnelle** —
la variable existe avant la décision prédite, mais **après le moment où mon
système doit répondre**. Un modèle qui l'utiliserait afficherait d'excellentes
performances en évaluation et serait inutilisable en production, faute de
disposer de son entrée principale.

La seule information dont un lycéen dispose au moment de choisir est la tension
**de l'année précédente**. C'est elle qui doit entrer dans le modèle, sous forme
de variable décalée.

## 4. Le décalage temporel est-il praticable ?

Une variable décalée suppose de retrouver, pour chaque formation d'une session,
la même formation à la session précédente. Je vérifie que la jointure est
possible avant de fonder la construction des variables dessus.

In [ ]:
def charger_avec_cle(annee: int) -> pl.DataFrame:
    """Charge un millésime en garantissant la présence de la clé de formation.

    Avant 2020, la colonne `cod_aff_form` n'existe pas dans le fichier publié.
    L'identifiant est toutefois présent dans le lien vers la fiche de la
    formation, sous le paramètre `g_ta_cod` — vérifié sur les millésimes
    récents, où les deux coïncident exactement.
    """
    d = charger(annee)
    if "cod_aff_form" not in d.columns:
        d = d.with_columns(
            pl.col("lien_form_psup")
            .str.extract(r"g_ta_cod=(\d+)")
            .cast(pl.Int64)
            .alias("cod_aff_form")
        )
    return d


for annee in (2018, 2019):
    d = charger_avec_cle(annee)
    avec_cle = d.filter(pl.col("cod_aff_form").is_not_null())
    unique = avec_cle["cod_aff_form"].n_unique() == avec_cle.height
    print(
        f"{annee} : {d.height} lignes | {avec_cle.height} avec clé "
        f"({avec_cle.height / d.height * 100:.1f} %) | clé unique : {unique}"
    )

print()
print("taux de jointure d'une session à la précédente")
for annee in range(2019, 2026):
    courante, precedente = charger_avec_cle(annee), charger_avec_cle(annee - 1)
    communes = set(courante["cod_aff_form"].drop_nulls().to_list()) & set(
        precedente["cod_aff_form"].drop_nulls().to_list()
    )
    print(f"  {annee - 1} -> {annee} : {len(communes):>6} formations retrouvées sur {courante.height:>6}  ({len(communes) / courante.height * 100:.1f} %)")

**Ce que j'en conclus.** Le décalage temporel est praticable sur l'ensemble de la
période, ce qui n'était pas acquis.

La colonne `cod_aff_form` est absente des fichiers 2018 et 2019. J'ai d'abord cru
devoir renoncer à ces deux millésimes, soit un quart de ma période
d'entraînement. L'identifiant s'y trouve en réalité **encodé dans le lien vers la
fiche de la formation**, sous le paramètre `g_ta_cod` : je l'ai vérifié sur les
millésimes récents, où la valeur extraite du lien et la colonne coïncident
exactement.

Une fois reconstruite, la clé est **unique partout où elle est extractible**, sans
aucun doublon, et couvre 92,4 % des lignes en 2018 et 94,6 % en 2019. Les lignes
sans lien se répartissent entre plusieurs filières, ce qui écarte l'hypothèse
d'une perte concentrée sur un type de formation.

Le taux de jointure d'une session à la suivante s'établit entre 82 % et 95 %. Les
formations non retrouvées sont celles qui ouvrent ou ferment d'une année sur
l'autre : c'est un phénomène réel, et non un défaut de méthode. Pour elles, le
modèle devra fonctionner sans historique — cas à traiter explicitement lors de la
construction des variables.

## Bilan

| Question | Ce que j'ai établi |
|---|---|
| Écart entre baccalauréats | médiane +8,5 points, mais 26,8 % des formations avantagent le bac professionnel |
| Structure de l'écart | BTS neutre (+1,1 pt) ; CPGE, BUT et PASS à médiane nulle pour le bac professionnel |
| Meilleur prédicteur | la tension : de 1,000 à 0,182 du quintile le plus détendu au plus tendu |
| Utilisabilité | la tension de l'année courante est une **fuite fonctionnelle** : elle n'existe pas quand mon système doit répondre |
| Solution | variable décalée d'une session, praticable — jointure de 82 % à 95 % selon la paire |

Deux décisions en découlent, que je consigne :

1. **Aucune variable mesurée après la formulation des vœux n'entre dans le
   modèle**, même si elle précède la décision d'admission. Le critère n'est pas
   la chronologie de la décision prédite, mais le moment où mon système doit
   répondre.
2. **Les variables de contexte sont décalées d'une session.** La construction des
   variables devra traiter explicitement les formations sans historique.

**La suite** : la féminisation par filière et la recherche des variables qui
prédisent le genre sans être le genre, puis la stabilité des séries entre 2018 et
2025.